### Core Strategy 1: Risk Parity with Omega

In [6]:
%time import pandas as pd
%time import numpy as np
%time import riskfolio as rf

%time import omega
%time from omega import start_loop, Stock, MarketOrder
%time from omega.objects import ScannerSubscription, TagValue
%time import logging

CPU times: user 5 μs, sys: 35 μs, total: 40 μs
Wall time: 208 μs
CPU times: user 3 μs, sys: 18 μs, total: 21 μs
Wall time: 21.9 μs
CPU times: user 4 μs, sys: 187 μs, total: 191 μs
Wall time: 388 μs
CPU times: user 3 μs, sys: 13 μs, total: 16 μs
Wall time: 17.9 μs
CPU times: user 8 μs, sys: 30 μs, total: 38 μs
Wall time: 392 μs
CPU times: user 4 μs, sys: 20 μs, total: 24 μs
Wall time: 24.1 μs
CPU times: user 2 μs, sys: 4 μs, total: 6 μs
Wall time: 6.91 μs


### Create an Omega trading app
This code creates an instance of an Omega trading app, connecting to the trading server at `127.0.0.1` and port `7497`. It also sets the client ID to 10 and specifies the account number as "DUE337747" for the trading session.

In [8]:
# You must run `start_loop` when using Omega from Jupyter Notebook
start_loop()

# To debug, instantiate Omega(log_level=logging.DEBUG)
app = omega.Omega("127.0.0.1", 7497, client_id=10, account="DUE337747")

API connection failed: ConnectionRefusedError(61, "Connect call failed ('127.0.0.1', 7497)")
Make sure API port on TWS/IBG is open


ConnectionRefusedError: [Errno 61] Connect call failed ('127.0.0.1', 7497)

### Universe selection
The `ScannerSubscription` defines the instrument to include in the scan, the location, and others. See `Omega.objects.ScannerSubscription` for details. Here's an example of creating a `ScannerSubscription`.

In [ ]:
scanner_subscription = ScannerSubscription(
    instrument="STK",
    location_code="STK.US.MAJOR",
    scan_code="HOT_BY_VOLUME",
    above_price=30,
    stock_type_filter="CORP"
)

With the `ScannerSubscrption` in place, you can further refine the results. Here's an example of building filters. These filters will refine the search results to include only those stocks with volume above 100,000 shares traded, market cap below 1,000,000,000 in local currency, and a price between 50 and 60 in local currency.

In [ ]:
filter_options = [
    TagValue("avgVolumeAbove", "10000000"),
    TagValue("marketCapAbove1e6", "100000"),  # in millions
    TagValue("currencyLike", "USD"),
]

Finally, we submit the scan through Omega. Interactive Brokers will return a list of contract objects by default or optionally string symbols. By returning the contract objects, we can immediately use the list to download historical data.

In [ ]:
hot_by_volume = app.scan(
    scanner_subscription, 
    return_as="contract",
    filter_options=filter_options
)

In [ ]:
hot_by_volume

### Data Preparation
Once we have our screened contracts, we use them to download historical stock price data. This data will be used for further analysis.

In [ ]:
prices = app.get_historical_data_for_many(
    contracts=hot_by_volume,
    end_date_time="",
    duration="1 Y",
    bar_size="1 day",
    what_to_show="MIDPOINT"
)

In [ ]:
prices

Omega returns the symbol of the requested contract in each row of data. This makes it easy to pivot the resulting DataFrame to put the closing prices of each contract in each column.

In [ ]:
returns = (
    prices
    .pivot(
        columns="symbol", 
        values="close"
    )
    .pct_change()
    .dropna(how="any")
)

In [ ]:
returns

### Portfolio Optimization
With the historical stock price data in hand, we can now proceed to portfolio optimization. We will use the riskfolio library to optimize the portfolio based on the Sharpe ratio, which measures the return of an investment compared to its risk. This code initializes a portfolio object using the riskfoliolib library, specifically with the returns data provided. The first line creates an instance of the Portfolio class with the given returns data. The second line sets the portfolio's lower return bound (lowerret) to 0.08%, which is likely used as a constraint in the portfolio optimization process to ensure that the expected returns do not fall below this threshold.

In [ ]:
port = rf.Portfolio(returns=returns)
port.lowerret = 0.0008

The `port.assets_stats` method in Riskfolio is used to compute the statistical properties of the assets in the portfolio. By setting `method_mu` to `"hist"`, it specifies that historical returns should be used to estimate the expected returns (`mu`). The `method_cov` parameter is set to `"ledoit"`, indicating that the Ledoit-Wolf shrinkage method should be used to estimate the covariance matrix of the returns, which helps to produce a more stable and reliable estimate.

In [ ]:
port.assets_stats(
    method_mu="hist", 
    method_cov="ledoit"
)

The `port.rp_optimization` method in Riskfolio performs risk parity optimization on the portfolio. By setting model to "Classic", it specifies the use of a traditional optimization model. The `rm` parameter is set to "MV", which indicates that Mean-Variance risk measure should be used. The `hist` parameter is set to `True`, meaning historical data is used for the optimization. The `rf` parameter is set to 0.05, indicating a risk-free rate of 5%. The resulting optimized portfolio weights are stored in the variable `w` as a pandas DataFrame.

In [ ]:
w = port.rp_optimization(
    model="Classic",
    rm="MV",
    hist=True,
    rf=0.05,
)

In [ ]:
w

The `rf.plot_risk_con` method in Riskfolio generates a plot showing the risk contributions of the assets in the portfolio. The variable `w` contains the portfolio weights obtained from the optimization. The `cov` parameter is set to the portfolio's covariance matrix (`port.cov`), and the returns parameter is set to the portfolio's returns (`port.returns`). The `rm` parameter is set to `"MV"`, specifying the use of Mean-Variance risk measure, and `rf` is set to `0.05`, indicating a 5% risk-free rate. The resulting plot, assigned to the variable `ax`, visualizes how each asset contributes to the overall risk of the portfolio.

In [ ]:
ax = rf.plot_risk_con(
    w,
    cov=port.cov,
    returns=port.returns,
    rm="MV",
    rf=0.05
)

### Rebalance the portfolio
Now that we have the optimal weights based on our screened universe, we can use Omega to ensure our portfolio reflects these weights. First, we'll divest the assets that are not currently in the screener. To do this, we can use the `app.positions_as_symbols` method from Omega. This will return a list of symbols currently in the portfolio

In [ ]:
positions = app.positions_as_symbols()
positions

We identify the positions that need to be divested from the current portfolio by comparing them to the optimized portfolio weights (`w`). We first calculate the set difference between the current positions and the indices of the optimized weights, resulting in a list of positions (`divest_`) that are not included in the optimized portfolio. Then, we create a DataFrame named `divest`, with these positions as the index, initialize their weights to zero, and labels the column as `"weights"`. This DataFrame represents the positions to be completely divested from the portfolio.

In [ ]:
divest_ = list(set(positions) - set(w.index))
divest = pd.DataFrame(
    index=divest_, 
    data=np.zeros(len(divest_)),
    columns=["weights"]
)

In [ ]:
divest

The code concatenates the divest DataFrame with the optimized portfolio weights (`w`) into a single DataFrame named `trade_weights`. We start with the assets to divest to free up margin before buying.

In [ ]:
trade_weights = pd.concat([divest, w])
trade_weights

We iterate over each row in the `trade_weights` DataFrame and create a Stock contract object with the given symbol, specifying "SMART" as the exchange and "USD" as the currency. Then, we place a market order using Omega's `app.order_target_percent` method to adjust the stock position to the target weight specified in the `trade_weights` DataFrame.

In [ ]:
for row in trade_weights.itertuples():
    print(f"Sending order for {row.Index}")
    contract = Stock(row.Index, "SMART", "USD")
    app.order_target_percent(contract=contract, order_type=MarketOrder, target=row.weights)

Confirm our fills.

In [ ]:
for fill in app.fills():
    print(fill.execution)

Finally, we disconnect our trading app to free up the client ID.

In [ ]:
app.disconnect()